In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
PROJECT_FILE = "pyproject.toml"

if not Path(PROJECT_FILE).exists():
    !git clone -q {REPOSITORY}
    %cd olot-course
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -e ".[ml]"

# 3.1 Convolutional neural networks (CNN)
---

## Motivation

* __Multilayer perceptron__
  - Treats the input as a vector of _independent_ features
  - Does not explicitly exploit spatial relationships between features
  - Suitable for tabular data
* __Convolutional neural network__
  - Exploits the _locality_ of the input data
  - Particularly useful for structured data such as images
  - Uses convolutional filters to detect local patterns
  - The same filters are applied across the input, reducing the number of parameters
  - Builds increasingly complex features by stacking convolutional layers

| A multilayer perceptron               |
| -----------------------               |
| <img src="figs/mlp.svg" width=400 />  |
| __Figure 1:__ A multilayer perceptron |

| A convolutional neural network |
| --------------------------     |
| ![](figs/cnn.svg)              |
| __Figure 2:__ A typical CNN architecture with two convolutional layers |

## 2D-Convolution Filtering

A convolutional layer applies a small filter (or _kernel_) across the input.

* A convolutional kernel is a small set of learnable weights that is reused across the input
* A technique to extract local features from the input
* The filter coefficients (weights) are learned during training
* Different filters can learn to detect different features

<center>
<div style="width: 460px;">

| A 2D convolution with a 3x3 kernel |
| ---------------------------------- |
| ![](figs/convolution.webp) |
| __Figure 3__: A 2D convolution applies a 3x3 filter across the input to produce a feature map. The filter (or kernel) is represented by the red square moving over the input. |

</div>
</center>

## &#128221; Example 3.1: Implementing a simple CNN

Let's implement the CNN architecture shown in Fig. 2

In [ ]:
import torch
import torch.nn as nn
from torchsummary import summary

class ConvolutionalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, padding=1, stride=2), 
            nn.ReLU(),
            nn.Conv2d(in_channels=6, out_channels=12, kernel_size=3, padding=1, stride=2), 
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(12*8*8,3)
        )

    def forward(self, x):
        return self.cnn(x)

and inspect the resulting architecture:

In [ ]:
model = ConvolutionalNet()
summary(model, input_size=(1, 32, 32))

## &#128221; Example 3.2: An application in volcanology

<center>
<div style="width: 600px;">
    
| Deep learning for seismicity classification |
| --- |
| ![](figs/fee2025_abstract.svg) |
| Fee, D. et al. (2025) __"A generalized deep learning model to detect and classify volcano seismicity"__, Volcanica, 8(1), pp. 305–323. doi: [10.30909/vol/rjss1878][doi] |

</div>
</center>

[doi]: https://doi.org/10.30909/vol/rjss1878

## &#128221; Example 3.3: Convolutional autoencoders

* An autoencoder is a neural network that learns to reconstruct its input
* It learns a compact representation of the input data:
  1. **Encoder:** compresses the input into a lower-dimensional representation
  2. **Decoder:** reconstructs the input from this representation
* The network is trained to minimise the reconstruction error

| Autoencoder diagram |
| ---- |
| ![](figs/autoencoder.svg) |
| __Figure 4__: An autoencoder consists of an encoder and a decoder. |

### Why are autoencoders useful?

Autoencoders can learn a **compact representation** of complex data.

| Application | Example in volcanology |
| :--- | --- |
| Denoising | Remove noise from seismic, infrasound, satellite, or other monitoring data while preserving relevant signals |
| Dimensionality reduction | Obtain compact representations of complex monitoring data or simulations |
| Anomaly detection | Identify unusual patterns in continuous monitoring data |
| Data compression | Efficiently represent large spatial or temporal datasets |
| Generative modelling | Generate new plausible data from the learned latent space |

## &#128221; Example 3.4: U-Net architecture

A U-Net has an encoder-decoder structure similar to an autoencoder, but uses skip connections to preserve fine-scale spatial information. Unlike a conventional autoencoder, its output is not necessarily a reconstruction of the input, making it particularly useful for image-to-image tasks.

| U-Net diagram |
| ---- |
| ![](figs/unet.svg) |
| __Figure 5__: A U-Net combines an encoder and decoder with skip connections that transfer spatial information between corresponding levels of the network. |

## &#128221; Example 3.5: More examples beyond volcanology

> * CNNs and autoencoders are not just tools for classifying volcano data,
> * they are general methods for extracting and reconstructing structure from complex spatial data

<center>
<div style="width: 600px;">
    
| Reducing the Dimensionality of Data with Neural Networks |
| --- |
| ![](figs/hinton.png) |
| __Figure 6__: G. E. Hinton, R. R. Salakhutdinov, Reducing the Dimensionality of Data with Neural Networks. _Science_ __313__, 504-507 (2006). DOI: [10.1126/science.1127647][hinton-doi] |

| Super-resolution reconstruction of brain MRIs |
| --- |
|![][brain] |
| __Figure 7__: Super-resolution reconstruction of brain magnetic resonance images via lightweight autoencoder. Extracted from [Andrew et al. (2021)][brain-doi] |

| Downscaling meteorological data |
| --- |
| ![][era5] |
| __Figure 8__: Downscaling based on deep learning applied to meteorological data. Extracted from [Tomasi et al. (2025)][era5-doi]|

</div>
</center>

[hinton-doi]: https://doi.org/10.1126/science.1127647
[brain]: https://ars.els-cdn.com/content/image/1-s2.0-S2352914821001945-gr4_lrg.jpg
[brain-doi]: https://doi.org/10.1016/j.imu.2021.100713
[era5]: https://gmd.copernicus.org/articles/18/2051/2025/gmd-18-2051-2025-f03-web.jpg
[era5-doi]: https://doi.org/10.5194/gmd-18-2051-2025